# Classical Phase Vocoder
This is an example phase vocoder implementation.

The phase vocoder requires a STFT.

In [23]:
import soundfile as sf
import numpy as np
import numpy.fft as fft
import math

def stft_frames(audio_frames, fft_size, hop_size):
    """
    Gets the number of STFT frames for given STFT parameters
    :param audio_frames: The number of audio frames
    :param fft_size: The FFT size
    :param hop_size: The hop size
    :return: The number of STFT frames
    """
    return 1 + (audio_frames - fft_size) // hop_size

def audio_frames(stft_frames, fft_size, hop_size):
    """
    Gets the number of audio frames for given STFT parameters
    :param stft_frames: The number of STFT frames
    :param fft_size: The FFT size
    :param hop_size: The hop size
    :return: The number of audio frames
    """
    return (stft_frames - 1) * hop_size + fft_size

def stft( input_sound, fft_size, hop_size, window):
    """
    Compute the short-time Fourier transform of a sound.
    Args:
        input_sound: 1D numpy array containing the sound samples
        fft_size: Size of the FFT to compute (number of frequency bins)
        hop_size: Number of samples to advance between successive frames
        window: 1D numpy array containing the analysis window (length = dft_size)
    Returns:
        f: 2D numpy array (frequencies x time) containing the complex-valued STFT
    """
    num_frames = stft_frames(np.max(input_sound.shape), fft_size, hop_size)
    spectrogram = np.zeros((fft_size // 2 + 1, num_frames), dtype=np.complex64)
    for frame in range(num_frames):
        start = frame * hop_size
        end = start + fft_size
        segment = input_sound[start:end] * window
        spectrum = np.fft.rfft(segment, n=fft_size)
        spectrogram[:, frame] = spectrum

    # Return a complex-valued spectrogram (frequencies x time)    
    return spectrogram

def istft( stft_output, fft_size, hop_size, window, norm=None):
    """Compute the inverse short-time Fourier transform of a spectrogram.
    Args:
        stft_output: 2D numpy array (frequencies x time) containing the complex-valued STFT
        fft_size: Size of the FFT that was computed (number of frequency bins)
        hop_size: Number of samples advanced between successive frames
        window: 1D numpy array containing the synthesis window (length = dft_size)
    Returns:
        x: 1D numpy array containing the reconstructed sound samples
    """
    number_frames = stft_output.shape[1]
    x = np.zeros((audio_frames(number_frames, fft_size, hop_size),))
    
    for frame in range(number_frames):
        start = frame * hop_size
        end = start + fft_size
        windowed_frame = np.fft.irfft(stft_output[:, frame], n=fft_size)
        x[start:end] += windowed_frame * window
    
    if norm is not None:
        return x / (norm + 1e-8)
    else:
        return x

def compute_window_norm(analysis_window, synthesis_window, hop_size, output_length):
    """
    Computes a window norm for the ISTFT
    :param analysis_window: The analysis window
    :param synthesis_window: The synthesis window
    :param num_frames: The number of frames in the STFT
    :return: The window norm array
    """
    analysis_norm = np.zeros((output_length,))
    synthesis_norm = np.zeros((output_length,))
    i = 0
    while (j := i + analysis_window.shape[-1]) <= output_length:
        analysis_norm[i:j] += analysis_window
        i += hop_size
    i = 0
    while (j := i + synthesis_window.shape[-1]) <= output_length:
        synthesis_norm[i:j] += synthesis_window
        i += hop_size
    norm = analysis_norm * synthesis_norm
    return norm


We'll start by reading in a signal.

In [24]:
audio, sr = sf.read("/home/jeff/data/recording/harmony.wav")
audio = audio.sum(axis=1)
audio.shape

(264600,)

Now we need to compute the STFT.

In [25]:
FFT_SIZE = 2048
HOP_SIZE = FFT_SIZE//4
window = np.hanning(FFT_SIZE)
analysis_data = stft(audio, FFT_SIZE, HOP_SIZE, window)
analysis_data.shape

(1025, 513)

We'll need to separate the magnitude and phase information for time stretching.

In [26]:
# The magnitude spectrogram
analysis_mag = np.abs(analysis_data)
# The phase spectrogram
analysis_phase = np.angle(analysis_data)

Now we need to compute the frequency matrix from our phase data. This is using the formulation from the Laroche/Dolson paper.
- $N$ is the FFT size
- $R_a$ is the analysis hop size and $R_s$ is the synthesis hop size
- $u$ is the hop index
- $0 \le k \le N/2$ is the frequency bin index
- $\Omega_k = \frac{2 \pi k}{N}$ is the center frequency of the $k$-th bin, in radians

The process goes as follows:
1. Compute the "heterodyned phase increment": $$ \Delta \Phi_k^u = \angle X(k, u) - \angle X(k, u-1) - R_a \Omega_k$$ (in other words, the phase difference minus the term at the end)
2. Take the "principal determination" $\Delta_p \Phi_k^u$ of the heterodyned phase increment (i.e. adjust it to be in the range $(-\pi, \pi)$)
3. Calculate the instantaneous frequency: $$ \hat{\omega}_k(u) = \Omega_k + \frac{1}{R_a}\Delta_p \Phi_k^u $$

When we're done, we'll have a frequency matrix of the same shape as our original phase matrix. This frequency matrix can be used to reconstruct the audio.

In [27]:
def wrap_pi(mx):
    """
    Wraps a matrix between -pi and pi.
    :param mx: The matrix to wrap
    :return: The wrapped matrix
    """
    mx += np.pi
    mx %= 2 * np.pi
    mx -= np.pi
    return mx

# 1. Compute phase diff matrix
phase_diff_mx = np.hstack((np.zeros((analysis_phase.shape[0], 1)), analysis_phase[:, :-1]))
phase_diff_mx = analysis_phase - phase_diff_mx

# 2. Compute heterodyned phase increment
omega = np.linspace(0, FFT_SIZE//2, FFT_SIZE//2+1) * 2 * np.pi / FFT_SIZE
omega = omega[:, np.newaxis]
delta_phi = wrap_pi(phase_diff_mx - HOP_SIZE * omega)
instantaneous_freq = omega + delta_phi / HOP_SIZE
instantaneous_freq.shape

(1025, 513)

We'll stretch by factor $\alpha$. Now the synthesis hop size $R_s = \alpha R_a$. (It doesn't matter that this doesn't nicely divide the FFT size.) We update the phase in the synthesis matrix using the formula
$$ \angle Y(k, u) = \angle Y(k, u-1) + R_s \hat{\omega}_k(u) $$

In [28]:
# 3. Now we're ready to time stretch.
alpha = 1.5
HOP_SIZE_SYNTHESIS = int(alpha * HOP_SIZE)

# 4. Compute the synthesis phase matrix
synthesis_phase = np.zeros(analysis_phase.shape)
synthesis_phase[:, 0] = HOP_SIZE_SYNTHESIS * instantaneous_freq[:, 0]
for i in range(1, synthesis_phase.shape[1]):
    synthesis_phase[:, i] = synthesis_phase[:, i-1] + HOP_SIZE_SYNTHESIS * instantaneous_freq[:, i]

# 5. Make the synthesis STFT matrix and render the audio.
synthesis_mx = analysis_mag * np.exp(1j * synthesis_phase)
synthesis_norm = compute_window_norm(window, window, HOP_SIZE_SYNTHESIS, audio_frames(synthesis_mx.shape[-1], FFT_SIZE, HOP_SIZE_SYNTHESIS))
out_audio = istft(synthesis_mx, FFT_SIZE, HOP_SIZE_SYNTHESIS, window, synthesis_norm)
# Get rid of click at beginning and end of output
fader = np.hanning(FFT_SIZE)
out_audio[:FFT_SIZE//2] *= fader[:FFT_SIZE//2]
out_audio[-(FFT_SIZE//2):] *= fader[-(FFT_SIZE//2):]

# Without phase locking, it's going to sound weirdly reverb-y.
sf.write("/home/jeff/recording/stretched.wav", out_audio, sr)

This result is ok, although the basic phase vocoder suffers from "phasiness" which we can resolve using phase locking.

We'd probably be happier if we could use the same hop size for synthesis as analysis. So we'll try interpolating the magnitude and frequency data. We need to be able to interpolate magnitude and frequency data in between analysis points.

In [29]:
def interp(frame: np.ndarray, frac: float):
    """
    Linearly interpolates a frame given a particular fractional index
    :param frame: The two adjacent frames to interpolate
    :param frac: The fractional index (0 < i < 1)
    :return: The interpolated frame
    """
    # For the first and last frame, just return as is
    if len(frame.shape) == 1:
        return frame
    else:
        slope = frame[:, 1] - frame[:, 0]
        return frame[:, 0] + slope * frac

Now we'll make an interpolated synthesis phase vocoder matrix.

In [30]:
interp_mag_frames = []
interp_freq_frames = []
f = 0
while f < analysis_mag.shape[-1]:
    if np.abs(f - round(f)) < 1e-3:
        interp_mag_frames.append(analysis_mag[:, round(f)])
        interp_freq_frames.append(instantaneous_freq[:, round(f)])
    elif f > analysis_mag.shape[-1]-1:
        interp_mag_frames.append(analysis_mag[:, -1])
        interp_freq_frames.append(instantaneous_freq[:, -1])
    else:
        i = math.floor(f)
        interp_mag_frames.append(interp(analysis_mag[:, i:i+2], f-i))
        interp_freq_frames.append(interp(instantaneous_freq[:, i:i+2], f-i))
    f += 1/alpha
interp_mag_frames = np.vstack(interp_mag_frames).T
interp_freq_frames = np.vstack(interp_freq_frames).T
interp_mag_frames.shape, analysis_mag.shape

((1025, 770), (1025, 513))

And we can follow the same process as before.

In [31]:
# 4. Compute the synthesis phase matrix
synthesis_phase = np.zeros(interp_freq_frames.shape)
synthesis_phase[:, 0] = HOP_SIZE * interp_freq_frames[:, 0]
for i in range(1, synthesis_phase.shape[1]):
    synthesis_phase[:, i] = synthesis_phase[:, i-1] + HOP_SIZE * interp_freq_frames[:, i]

# 5. Make the synthesis STFT matrix and render the audio.
synthesis_mx = interp_mag_frames * np.exp(1j * synthesis_phase)
synthesis_norm = compute_window_norm(window, window, HOP_SIZE, audio_frames(synthesis_mx.shape[-1], FFT_SIZE, HOP_SIZE))
out_audio = istft(synthesis_mx, FFT_SIZE, HOP_SIZE, window, synthesis_norm)
# Get rid of click at beginning and end of output
fader = np.hanning(FFT_SIZE)
out_audio[:FFT_SIZE//2] *= fader[:FFT_SIZE//2]
out_audio[-(FFT_SIZE//2):] *= fader[-(FFT_SIZE//2):]

# Without phase locking, it's going to sound weirdly reverb-y.
sf.write("/home/jeff/recording/stretched2.wav", out_audio, sr)

This should also work with longer stretches, unlike the original method which will require lots of overlap.

In [32]:
alpha=4
interp_mag_frames = []
interp_freq_frames = []
f = 0
while f < analysis_mag.shape[-1]:
    if np.abs(f - round(f)) < 1e-3:
        interp_mag_frames.append(analysis_mag[:, round(f)])
        interp_freq_frames.append(instantaneous_freq[:, round(f)])
    elif f > analysis_mag.shape[-1]-1:
        interp_mag_frames.append(analysis_mag[:, -1])
        interp_freq_frames.append(instantaneous_freq[:, -1])
    else:
        i = math.floor(f)
        interp_mag_frames.append(interp(analysis_mag[:, i:i+2], f-i))
        interp_freq_frames.append(interp(instantaneous_freq[:, i:i+2], f-i))
    f += 1/alpha
interp_mag_frames = np.vstack(interp_mag_frames).T
interp_freq_frames = np.vstack(interp_freq_frames).T
# 4. Compute the synthesis phase matrix
synthesis_phase = np.zeros(interp_freq_frames.shape)
synthesis_phase[:, 0] = HOP_SIZE * interp_freq_frames[:, 0]
for i in range(1, synthesis_phase.shape[1]):
    synthesis_phase[:, i] = synthesis_phase[:, i-1] + HOP_SIZE * interp_freq_frames[:, i]

# 5. Make the synthesis STFT matrix and render the audio.
synthesis_mx = interp_mag_frames * np.exp(1j * synthesis_phase)
synthesis_norm = compute_window_norm(window, window, HOP_SIZE, audio_frames(synthesis_mx.shape[-1], FFT_SIZE, HOP_SIZE))
out_audio = istft(synthesis_mx, FFT_SIZE, HOP_SIZE, window, synthesis_norm)
# Get rid of click at beginning and end of output
fader = np.hanning(FFT_SIZE)
out_audio[:FFT_SIZE//2] *= fader[:FFT_SIZE//2]
out_audio[-(FFT_SIZE//2):] *= fader[-(FFT_SIZE//2):]

# Without phase locking, it's going to sound weirdly reverb-y.
sf.write("/home/jeff/recording/stretched3.wav", out_audio, sr)